# Capstone Report: Scene Classification, Interpretability, and Deployment Readiness

## 1. Proposal

**Problem.** The project addresses multiclass natural scene classification: given an image, predict whether it belongs to one of six scene categories: buildings, forest, glacier, mountain, sea, or street.

**Why it matters.** Scene classification is a practical computer vision task for photo organization, mobile camera assistance, environmental monitoring, field documentation, and low-connectivity visual triage. Even when the application is not high-stakes, the model must be evaluated carefully because scene images can contain people, homes, vehicles, addresses, and location clues.

**Data.** The project uses the Intel Image Classification dataset. The training source contains 14,034 labeled images. A validation split of 2,105 images is taken from the training source, leaving 11,929 training images. The test set contains 3,000 labeled images.

**Model choice.** The implemented baseline is a compact TensorFlow/Keras CNN with image resizing, augmentation, rescaling, convolution blocks, batch normalization, dropout, AdamW optimization, checkpointing, and TensorFlow Lite export. The recommended next model is MobileNetV2 or MobileNetV3 transfer learning because it better aligns with the edge-deployment goal.

**Success criteria.** The prototype target was at least 70% test accuracy and 0.70 macro F1. A stricter deployment gate is: test accuracy >= 0.80, macro F1 >= 0.78, every class recall >= 0.70, and TFLite size <= 5 MB.

## 2. Hypothesis

**Primary hypothesis:** For the Intel natural scene classification task, a convolutional neural network will learn spatial texture and shape features that classify six scene categories with at least **70% test accuracy** and **0.70 macro F1**.

**Independent variables:** model family, input resolution, preprocessing, augmentation strength, optimizer, regularization, training schedule, and deployment format.

**Dependent variables:** test accuracy, macro F1, per-class recall, confusion-matrix error patterns, exported model size, and expected edge-deployment feasibility.

**Anticipated relationship:** CNN-based models should outperform sequence-oriented approaches for still images because convolution preserves local two-dimensional structure and builds hierarchical visual features from edges to textures to scene-level patterns. Transfer-learning backbones such as MobileNetV2 or EfficientNet are expected to improve class-level generalization because ImageNet pretraining provides reusable low- and mid-level visual features.

**Deployment hypothesis:** The current scratch CNN should be treated as a prototype unless it satisfies all deployment gates. Based on the results, the prototype reaches the basic learning target but does **not** satisfy deployment readiness because minimum per-class recall is below 0.70.

## 3. Empirical Strategy

| Stage | Design choice | Rationale | Constraint or mitigation |
|---|---|---|---|
| Dataset | Use the local Intel scene dataset in `data/seg_train`, `data/seg_test`, and `data/seg_pred`. | The folder structure provides a clean six-class image classification task. | Keep train/test folders separate to reduce leakage. |
| Split | Use train, validation, and test evaluation. | Validation guides checkpoints; test set is reserved for final evaluation. | Do not tune architecture on test performance. |
| Preprocessing | Resize images to 128x128 and rescale pixel values. | Gives the CNN a stable input contract. | Lower resolution if memory or latency constraints dominate. |
| Augmentation | Apply random flip, rotation, and zoom during training. | Improves robustness to camera variation. | Avoid transformations that change semantic class labels. |
| Model | Train compact CNN baseline; recommend MobileNet transfer learning next. | CNNs match image structure; MobileNet supports edge deployment. | Compare accuracy, model size, latency, and weak-class recall. |
| Optimization | AdamW, batch normalization, dropout, checkpointing. | Stabilizes learning and reduces overfitting. | Use validation checkpoint rather than final epoch only. |
| Evaluation | Accuracy, macro F1, per-class recall, confusion matrix. | Aggregate accuracy hides class-specific failures. | Require per-class recall floors before deployment. |
| Interpretation | Use Grad-CAM/failure review as the chosen interpretability plan. | Tests whether the model attends to meaningful scene evidence. | Reject models that rely on shortcuts such as sky color or image borders. |
| Deployment | Export `.keras`, SavedModel, and `.tflite`. | Tests portability beyond the notebook. | Benchmark on target hardware before release. |

## 4. Architecture Design and Alternatives

| Candidate | Strength | Decision |
|---|---|---|
| RNN/GRU over image rows | Useful as a learning comparison. | Rejected because row-wise sequences do not preserve two-dimensional spatial locality as directly as convolution. |
| Scratch compact CNN | Transparent, small, trains quickly, exports cleanly to TFLite. | Accepted as the implemented baseline. It is useful for learning and deployment testing but not final-release quality. |
| ResNet-50 | Strong general-purpose image baseline. | Rejected for the first deployment target because it is larger than needed for an edge-focused prototype. |
| MobileNetV2/MobileNetV3 | Designed for mobile and embedded vision with depthwise separable convolutions. | Preferred next model because it directly matches the deployment hypothesis: better accuracy than scratch CNN while preserving small size and low latency. |
| EfficientNet/EfficientNet-Lite | Strong accuracy/efficiency scaling. | Good second transfer-learning comparison, but slightly less direct than MobileNet for the first TinyML-oriented iteration. |
| Vision Transformer | Powerful when large-scale pretraining and compute are available. | Rejected for this stage because the dataset size and edge constraints favor CNN/mobile backbones. |

**Final architecture position:** The scratch CNN is the baseline implementation. The next recommendation is MobileNet-family transfer learning as the upgrade because it balances state-of-the-art transfer learning with deployment constraints.

## 5. Optimization and Results

The baseline optimization strategy used several complementary techniques: AdamW for adaptive optimization with weight decay, batch normalization for stable convolutional training, dropout for classifier regularization, data augmentation for robustness, checkpointing to preserve the best validation model, and TensorFlow Lite conversion to test deployment feasibility.

**Training dynamics.** Validation accuracy peaked at epoch 3 with validation accuracy 0.740. The final epoch reached training accuracy 0.792 and validation accuracy 0.701. This pattern suggests the model learned useful scene features but began to show diminishing returns and overfitting risk.

**Test results.**

- Test accuracy: **0.724**
- Test loss: **0.746**
- Macro F1: **0.723**
- Minimum per-class recall: **0.424**
- TFLite model size: **0.291 MB**

| Class | Precision | Recall | F1 | Support |
|---|---:|---:|---:|---:|
| buildings | 0.656 | 0.824 | 0.730 | 437 |
| forest | 0.982 | 0.812 | 0.889 | 474 |
| glacier | 0.582 | 0.863 | 0.695 | 553 |
| mountain | 0.759 | 0.642 | 0.696 | 525 |
| sea | 0.927 | 0.424 | 0.581 | 510 |
| street | 0.708 | 0.794 | 0.749 | 501 |
| macro avg | 0.769 | 0.726 | 0.723 | 3000 |
| weighted avg | 0.767 | 0.724 | 0.721 | 3000 |


The model clears the basic prototype target: accuracy is above 70% and macro F1 is above 0.70. It does not clear the stricter deployment gate because `sea` recall is only 0.424. That means many true sea images are missed even though sea precision is high. The most important next optimization is not simply more epochs; it is transfer learning plus targeted failure analysis for weak classes.

## 6. Results Visualizations


**Training curve:** validation performance improves early and then weakens, indicating diminishing returns and possible overfitting in the scratch CNN.

![Training curve](../artifacts/milestone4_training_curve.png)

**Confusion matrix:** the matrix highlights which scene categories are confused. The most important operational issue is low recall for `sea`, meaning many sea images are classified as other classes.

![Confusion matrix](../artifacts/milestone4_confusion_matrix.png)

**Sample batch:** the sample image grid documents the visual diversity of the scene dataset and shows why class overlap is plausible, especially between glacier/mountain/sea images with horizon and sky regions.

![Sample batch](../artifacts/milestone4_sample_batch.png)

The next iteration should prioritize class-level recall and failure-case review rather than optimizing aggregate accuracy alone.

## 7. Interpretability Plan

**Chosen method: Grad-CAM.** Grad-CAM is appropriate because the implemented model is a CNN and the decision context requires visual evidence. It produces a class-discriminative heatmap over the final convolutional feature map, helping determine whether the model focuses on meaningful regions such as water, road structure, tree canopy, mountain ridges, ice texture, or building edges.

| Method | Benefit | Limitation | Decision |
|---|---|---|---|
| Grad-CAM | Shows class-relevant image regions for CNNs. | Coarse localization. | Best primary method for this model and stakeholder report. |
| Saliency maps | Pixel-level sensitivity. | Often noisy for natural scenes. | Secondary diagnostic only. |
| LIME | Model-agnostic explanation. | Depends heavily on image segmentation choices and is slower. | Useful if comparing Keras and TFLite as black boxes. |
| SHAP | Strong contribution framework. | Expensive for images and less practical for this notebook. | Not the first choice for this capstone. |

If Grad-CAM repeatedly highlights shortcuts such as blank sky, borders, watermarks, or camera artifacts, the model should not be deployed without dataset cleanup, stronger augmentation, or a transfer-learning comparison. Explanations should support model debugging, not be treated as proof that the model is correct.

## 8. Risk Reflection Mapped to Model Behavior

| Risk | Concrete failure mode | Mitigation |
|---|---|---|
| Domain shift | The model sees unfamiliar geography, night images, fog, rural roads, different cameras, or seasonal conditions. | Evaluate slices by region, lighting, season, and camera source before release. |
| Class brittleness | `sea` recall is below the deployment floor, so many true sea images are missed. | Do not deploy the current model as final; improve with transfer learning and class-targeted evaluation. |
| Automation bias | Users may over-trust confident labels. | Show confidence, suppress low-confidence outputs, and use human review for consequential decisions. |
| Shortcut learning | The CNN may rely on sky color, horizon lines, borders, or artifacts rather than scene evidence. | Use Grad-CAM and misclassified examples to inspect model behavior. |
| Privacy | Scene images may include people, homes, vehicles, addresses, or location clues. | Prefer local TFLite inference and avoid uploading raw images by default. |
| Accountability gap | Edge devices can fail silently if there is no monitoring. | Use privacy-preserving logs: model version, confidence bands, latency, and opt-in error reports. |
| Energy and cost | Repeated large experiments increase compute waste. | Use compact baselines, pretrained backbones, checkpointing, and targeted ablations. |
| Misuse | Scene labels could support surveillance or unauthorized location inference. | Limit scope, document prohibited uses, and avoid high-stakes identification applications. |

## 9. Stakeholder and Sociotechnical Reflection

| Stakeholder | Benefit | Potential harm | Design response |
|---|---|---|---|
| End users | Fast scene labels, potentially without uploading images. | Misclassification, overconfidence, privacy exposure. | Local inference, confidence thresholds, and clear uncertainty handling. |
| Affected communities | Better support for low-connectivity or field settings. | Geographic bias may reduce performance in underrepresented environments. | Audit across regions, weather, camera quality, and settlement types. |
| Deployers | Lightweight model can run on mobile or edge hardware. | They may assume test accuracy generalizes to field data. | Require hardware benchmarks and field validation before release. |
| Regulators/reviewers | Documented limitations and measurable release gates. | Lack of traceability if edge devices are unmanaged. | Versioning, model cards, data documentation, and privacy-preserving logs. |
| Developers | Reusable training and export pipeline. | Pressure to optimize aggregate accuracy while ignoring weak classes. | Use macro F1, per-class recall, confusion matrix, and Grad-CAM checks as acceptance criteria. |

Local inference improves privacy and latency, but it can reduce centralized visibility into failures. The responsible compromise is to avoid raw-image logging by default while enabling consent-based error reporting and privacy-preserving telemetry.

## 10. Limitations and Improvements

**Current limitations:**

- The scratch CNN does not meet deployment thresholds, mainly because the weakest class recall is too low.
- The model has not yet been benchmarked on a real mobile or embedded device.
- The report identifies Grad-CAM as the interpretability method, but the final decision should include saved examples for both correct and incorrect predictions.
- The current model is trained from scratch, so it may need more data or stronger representations than the dataset can support alone.

**Next improvements:**

- Train a MobileNetV2 or MobileNetV3 transfer-learning model and compare it with the scratch CNN.
- Add a small ablation table for augmentation, dropout, and input size.
- Add misclassified-example panels with Grad-CAM overlays.
- Quantize the TFLite model and benchmark latency, memory, and accuracy change on target hardware.
- Define a confidence threshold for abstention or human review.